# Analyzing DHS microdata

Nigeria 2018 - J:\DATA\DHS_PROG_DHS\NGA\2018
India 2015-2016 - J:\DATA\DHS_PROG_DHS\IND\2015_2016

More recent is available, but probably weird due to COVID

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [1]:
import pandas as pd, numpy as np
from lsff_utils import data_processing

%load_ext autoreload
%autoreload 2

!date

Wed Aug 14 14:06:02 PDT 2024


In [2]:
location = "india"

In [3]:
# Parameters
location = "ethiopia"

## Load data, name columns

In [4]:
directory = {
    "india": "/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/",
    "nigeria": "/snfs1/DATA/DHS_PROG_DHS/NGA/2018/",
    "ethiopia": "/snfs1/DATA/DHS_PROG_DHS/ETH/2016/",
}[location]

### WRA

In [5]:
%%time

wra_data_file_name = {
    "india": "IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA",
    "nigeria": "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA",
    "ethiopia": "ETH_DHS7_2016_WN_ETIR71FL_Y2019M12D11.DTA",
}[location]

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
    "v213": "currently_pregnant",
}
wra_data = pd.read_stata(
    directory + wra_data_file_name,
    columns=wra_columns.keys(),
)

CPU times: user 722 ms, sys: 359 ms, total: 1.08 s
Wall time: 1.54 s


In [6]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [7]:
wra_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    wra_data.wealth_quintile
)

In [8]:
wra_data["pregnant"] = wra_data.currently_pregnant.str.strip().map(
    {
        "not pregnant, don't know": "not_pregnant",
        "no or unsure": "not_pregnant",
        "pregnant": "pregnant",
        "yes": "pregnant",
    }
)

In [9]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [10]:
birth_data_file_name = {
    "india": "IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA",
    "nigeria": "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA",
    "ethiopia": "ETH_DHS7_2016_BR_ETBR71FL_Y2019M12D11.DTA",
}[location]

birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
}
if location == "india":
    birth_columns["s220a"] = "duration_of_pregnancy"
else:
    birth_columns["b20"] = "duration_of_pregnancy"

birth_data = pd.read_stata(
    directory + birth_data_file_name,
    columns=birth_columns.keys(),
)
birth_data

,v005,v008,v190,b3,m18,m19,b20
0,5087433,1305,poorer,1274,average,not weighed at birth,9.0
1,5087433,1305,poorer,1261,average,not weighed at birth,9.0
2,5087433,1305,poorer,1250,larger than average,not weighed at birth,9.0
3,5087433,1305,poorer,1239,NaN,NaN,9.0
4,5087433,1305,poorer,1216,NaN,NaN,NaN
...,...,...,...,...,...,...,...
41387,685395,1301,richest,1087,NaN,NaN,NaN
41388,685395,1301,richest,1299,very small,2000.0,9.0
41389,685395,1301,richest,1295,average,don't know,9.0
41390,685395,1301,richest,1158,NaN,NaN,NaN


In [11]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    birth_data.wealth_quintile
)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date,size_of_child,birth_weight_kilograms,duration_of_pregnancy
0,5.087433,1305,2,1274,average,not weighed at birth,9.0
1,5.087433,1305,2,1261,average,not weighed at birth,9.0
2,5.087433,1305,2,1250,larger than average,not weighed at birth,9.0
3,5.087433,1305,2,1239,NaN,NaN,9.0
4,5.087433,1305,2,1216,NaN,NaN,NaN
...,...,...,...,...,...,...,...
41387,0.685395,1301,5,1087,NaN,NaN,NaN
41388,0.685395,1301,5,1299,very small,2000.0,9.0
41389,0.685395,1301,5,1295,average,don't know,9.0
41390,0.685395,1301,5,1158,NaN,NaN,NaN


### Household members

In [12]:
%%time

hhm_data_file_name = {
    "india": "IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA",
    "nigeria": "NGA_DHS7_2018_HHM_NGPR7AFL_Y2019M11D05.DTA",
    "ethiopia": "ETH_DHS7_2016_HHM_ETPR71FL_Y2019M12D11.DTA",
}[location]

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw_adult",
    "ha56": "hemoglobin_adjusted_adult",
    "ha57": "anemia_adult",
    "hc53": "hemoglobin_raw_child",
    "hc56": "hemoglobin_adjusted_child",
    "hc57": "anemia_child",
}
hhm_data = pd.read_stata(
    directory + hhm_data_file_name,
    columns=hhm_columns.keys(),
)
hhm_data

CPU times: user 141 ms, sys: 115 ms, total: 255 ms
Wall time: 488 ms


,hv001,hv002,hv005,hv008,hvidx,hv105,hv104,ha0,ha1,hv270,ha53,ha56,ha57,hc53,hc56,hc57
0,1,15,5056023,1305,1,60,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
1,1,15,5056023,1305,2,55,female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
2,1,15,5056023,1305,3,20,male,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
3,1,15,5056023,1305,4,9,female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
4,1,15,5056023,1305,5,7,female,NaN,NaN,middle,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75219,645,464,664627,1301,2,25,female,2.0,25.0,richest,110.0,100.0,mild,NaN,NaN,NaN
75220,645,464,664627,1301,3,0,female,NaN,NaN,richest,NaN,NaN,NaN,NaN,NaN,NaN
75221,645,464,664627,1301,4,18,female,4.0,18.0,richest,132.0,122.0,not anemic,NaN,NaN,NaN
75222,645,485,664627,1301,1,25,male,NaN,NaN,richest,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [14]:
id_columns = ["cluster_number", "household_number", "line_number"]
hhm_data = hhm_data.merge(
    wra_data[id_columns + ["pregnant"]],
    on=id_columns,
    how="left",
)
hhm_data["pregnant"] = hhm_data.pregnant.fillna("not_pregnant")

In [15]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

In [16]:
hhm_data["sex"] = hhm_data.sex.str.title()

In [17]:
# Interesting -- sometimes age is quite off between hemoglobin and base.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

,cluster_number,household_number,weight,date_of_interview,line_number,age,sex,index_to_household,age_hemoglobin,wealth_quintile,hemoglobin_raw_adult,hemoglobin_adjusted_adult,anemia_adult,hemoglobin_raw_child,hemoglobin_adjusted_child,anemia_child,pregnant
60428,522,337,2635159,1301,2,27.0,Female,2.0,44.0,richest,116.0,112.0,mild,NaN,NaN,NaN,not_pregnant
7312,64,130,170527,1301,2,18.0,Female,2.0,35.0,richest,121.0,116.0,mild,NaN,NaN,NaN,not_pregnant
65314,563,483,136418,1302,2,30.0,Female,2.0,45.0,poorer,105.0,103.0,mild,NaN,NaN,NaN,not_pregnant
63528,549,97,85401,1303,2,30.0,Female,2.0,45.0,richest,139.0,139.0,not anemic,NaN,NaN,NaN,not_pregnant
4649,40,126,2516444,1301,2,26.0,Female,2.0,40.0,richer,not present,NaN,NaN,NaN,NaN,NaN,not_pregnant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75216,645,449,664627,1301,3,11.0,Male,NaN,NaN,richest,NaN,NaN,NaN,NaN,NaN,NaN,not_pregnant
75217,645,449,664627,1301,4,0.0,Female,NaN,NaN,richest,NaN,NaN,NaN,NaN,NaN,NaN,not_pregnant
75218,645,464,664627,1301,1,32.0,Male,NaN,NaN,richest,NaN,NaN,NaN,NaN,NaN,NaN,not_pregnant
75220,645,464,664627,1301,3,0.0,Female,NaN,NaN,richest,NaN,NaN,NaN,NaN,NaN,NaN,not_pregnant


In [18]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

count    17137.000000
mean        -0.133396
std          1.368562
min        -17.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         17.000000
dtype: float64

In [19]:
age_bin_edges = [0, 5, 15, 30, 50, 125]
age_group = pd.IntervalIndex(
    pd.cut(hhm_data.age, age_bin_edges, right=False, include_lowest=True)
)
hhm_data["age_start"] = age_group.left
hhm_data["age_end"] = age_group.right

In [20]:
hhm_data["wealth_quintile"] = data_processing.recode_dhs_wealth_quintile(
    hhm_data.wealth_quintile
)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [21]:
for type in ["child", "adult"]:
    for base_col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
        col = f"{base_col}_{type}"
        hhm_data[col] = (
            hhm_data[col]
            .astype(str)
            .replace(
                {
                    "not tested": np.nan,
                    "not present": np.nan,
                    "refused": np.nan,
                    "other": np.nan,
                }
            )
            .astype(float)
        )

In [22]:
for base_col in ["hemoglobin_raw", "hemoglobin_adjusted", "anemia"]:
    assert (hhm_data.filter(like=base_col).notnull().sum(axis=1) <= 1).all()
    hhm_data[base_col] = np.nan
    # Could use bfill instead of this loop, but it was incredibly slow for me
    for col in hhm_data.filter(like=base_col).columns:
        hhm_data[base_col] = hhm_data[base_col].fillna(hhm_data[col])

In [23]:
assert (
    hhm_data[(hhm_data.sex == "male") & (hhm_data.age > 5)]
    .hemoglobin_raw.isnull()
    .all()
)

## Hemoglobin

In [24]:
other_overlapping_columns = (
    (set(wra_data.columns) & set(hhm_data.columns)) - set(id_columns) - {"weight"}
)
other_overlapping_columns

{'pregnant', 'wealth_quintile'}

In [25]:
wra_hhm_joined = wra_data.merge(
    hhm_data.drop(columns=["weight"]),
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

,cluster_number,household_number,line_number,weight,interview_date,date_of_birth,wealth_quintile_wra,currently_pregnant,pregnant_wra,date_of_interview,...,anemia_adult,hemoglobin_raw_child,hemoglobin_adjusted_child,anemia_child,pregnant_hhm,age_start,age_end,hemoglobin_raw,hemoglobin_adjusted,anemia
0,1,17,2,5.087433,1305,846,2,no or unsure,not_pregnant,1305,...,not anemic,NaN,NaN,NaN,not_pregnant,30.0,50.0,144.0,136.0,not anemic
1,1,17,3,5.087433,1305,1097,2,no or unsure,not_pregnant,1305,...,not anemic,NaN,NaN,NaN,not_pregnant,15.0,30.0,130.0,122.0,not anemic
2,1,18,2,5.087433,1305,798,2,no or unsure,not_pregnant,1305,...,not anemic,NaN,NaN,NaN,not_pregnant,30.0,50.0,135.0,127.0,not anemic
3,1,25,2,5.087433,1305,751,2,no or unsure,not_pregnant,1305,...,not anemic,NaN,NaN,NaN,not_pregnant,30.0,50.0,128.0,120.0,not anemic
4,1,25,10,5.087433,1305,1093,2,no or unsure,not_pregnant,1305,...,NaN,NaN,NaN,NaN,not_pregnant,15.0,30.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15678,645,419,4,0.685395,1301,1097,5,no or unsure,not_pregnant,1301,...,not anemic,NaN,NaN,NaN,not_pregnant,15.0,30.0,147.0,137.0,not anemic
15679,645,449,2,0.685395,1301,895,5,no or unsure,not_pregnant,1301,...,not anemic,NaN,NaN,NaN,not_pregnant,30.0,50.0,162.0,152.0,not anemic
15680,645,464,2,0.685395,1301,1001,5,no or unsure,not_pregnant,1301,...,mild,NaN,NaN,NaN,not_pregnant,15.0,30.0,110.0,100.0,mild
15681,645,464,4,0.685395,1301,1079,5,no or unsure,not_pregnant,1301,...,not anemic,NaN,NaN,NaN,not_pregnant,15.0,30.0,132.0,122.0,not anemic


In [26]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f"{col}_wra"] == wra_hhm_joined[f"{col}_hhm"]).all()
    wra_hhm_joined[col] = wra_hhm_joined[f"{col}_wra"]
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f"{col}_wra", f"{col}_hhm"])

In [27]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values - average) ** 2, weights=weights)
    return pd.Series(
        {
            "mean": average,
            "sd": np.sqrt(variance),
            # https://ngreifer.github.io/WeightIt/reference/ESS.html
            "effective_sample_size": (weights.sum() ** 2) / (weights**2).sum(),
        }
    )

In [28]:
pregnant_with_anemia_status = wra_hhm_joined[
    (wra_hhm_joined.pregnant == "pregnant") & wra_hhm_joined.anemia.notnull()
]

In [29]:
# Matches table 10.21.1
pregnant_with_anemia_status.weight.sum()

1087.537222

In [30]:
# Within rounding error of table 10.21.1 value
weighted_avg_and_std(
    pregnant_with_anemia_status.anemia == "severe",
    pregnant_with_anemia_status.weight,
)

mean                       0.022009
sd                         0.146711
effective_sample_size    459.276654
dtype: float64

In [31]:
# Within rounding error of table 10.21.1 value for any anemia
weighted_avg_and_std(
    pregnant_with_anemia_status.anemia.isin(["severe", "moderate", "mild"]),
    pregnant_with_anemia_status.weight,
)

mean                       0.291341
sd                         0.454380
effective_sample_size    459.276654
dtype: float64

In [32]:
assert (
    (pregnant_with_anemia_status.anemia == "severe")
    == (pregnant_with_anemia_status.hemoglobin_adjusted < 70)
).all()

In [33]:
assert (
    (pregnant_with_anemia_status.anemia.isin(["severe", "moderate", "mild"]))
    == (pregnant_with_anemia_status.hemoglobin_adjusted < 110)
).all()

In [34]:
adult_hemoglobin_disparities = (
    wra_hhm_joined.groupby(["wealth_quintile", "pregnant"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
adult_hemoglobin_disparities

mean         sd  effective_sample_size
wealth_quintile pregnant                                                  
1               not_pregnant  124.952266  20.051752            1107.366224
                pregnant      113.492095  22.155564             114.663696
2               not_pregnant  128.293821  17.054204            1034.823622
                pregnant      116.926741  19.180831              97.659958
3               not_pregnant  129.201329  16.563451            1158.806421
                pregnant      118.841395  14.629155              84.592517
4               not_pregnant  130.193045  15.973793            1214.846084
                pregnant      116.357764  15.836060              86.664891
5               not_pregnant  131.983786  14.994380            1698.175645
                pregnant      122.549570  15.731898              78.744645

In [35]:
adult_hemoglobin_disparities = adult_hemoglobin_disparities.reset_index()
adult_hemoglobin_disparities = pd.concat(
    [
        adult_hemoglobin_disparities.assign(sex="Female", age_start=15, age_end=125),
        # Assumption: males are like non-pregnant WRA
        adult_hemoglobin_disparities[
            adult_hemoglobin_disparities.pregnant == "not_pregnant"
        ].assign(sex="Male", age_start=15, age_end=125),
    ]
)
adult_hemoglobin_disparities = adult_hemoglobin_disparities.set_index(
    ["sex", "age_start", "age_end", "pregnant", "wealth_quintile"]
)
adult_hemoglobin_disparities

mean         sd  \
sex    age_start age_end pregnant     wealth_quintile                          
Female 15        125     not_pregnant 1                124.952266  20.051752   
                         pregnant     1                113.492095  22.155564   
                         not_pregnant 2                128.293821  17.054204   
                         pregnant     2                116.926741  19.180831   
                         not_pregnant 3                129.201329  16.563451   
                         pregnant     3                118.841395  14.629155   
                         not_pregnant 4                130.193045  15.973793   
                         pregnant     4                116.357764  15.836060   
                         not_pregnant 5                131.983786  14.994380   
                         pregnant     5                122.549570  15.731898   
Male   15        125     not_pregnant 1                124.952266  20.051752   
                                      2                128.293821  17.054204   
                                      3                129.201329  16.563451   
                                      4                130.193045  15.973793   
                                      5                131.983786  14.994380   

                                                       effective_sample_size  
sex    age_start age_end pregnant     wealth_quintile                         
Female 15        125     not_pregnant 1                          1107.366224  
                         pregnant     1                           114.663696  
                         not_pregnant 2                          1034.823622  
                         pregnant     2                            97.659958  
                         not_pregnant 3                          1158.806421  
                         pregnant     3                            84.592517  
                         not_pregnant 4                          1214.846084  
                         pregnant     4                            86.664891  
                         not_pregnant 5                          1698.175645  
                         pregnant     5                            78.744645  
Male   15        125     not_pregnant 1                          1107.366224  
                                      2                          1034.823622  
                                      3                          1158.806421  
                                      4                          1214.846084  
                                      5                          1698.175645

In [36]:
child_hemoglobin_disparities = (
    hhm_data[(hhm_data.age <= 5)]
    .assign(age_start=0, age_end=5, pregnant="not_pregnant")
    .groupby(["sex", "age_start", "age_end", "pregnant", "wealth_quintile"])
    .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
    .sort_index()
)
child_hemoglobin_disparities

mean         sd  \
sex    age_start age_end pregnant     wealth_quintile                          
Female 0         5       not_pregnant 1                100.844346  19.057214   
                                      2                104.929413  16.388767   
                                      3                108.144135  15.798272   
                                      4                106.092684  14.667368   
                                      5                110.229853  14.279931   
Male   0         5       not_pregnant 1                 99.549536  18.535592   
                                      2                105.371341  16.208981   
                                      3                107.209986  14.817859   
                                      4                106.623690  17.780204   
                                      5                108.247030  15.043161   

                                                       effective_sample_size  
sex    age_start age_end pregnant     wealth_quintile                         
Female 0         5       not_pregnant 1                           492.680769  
                                      2                           409.805632  
                                      3                           379.924099  
                                      4                           369.991690  
                                      5                           275.680329  
Male   0         5       not_pregnant 1                           499.512573  
                                      2                           446.163156  
                                      3                           415.615837  
                                      4                           353.032739  
                                      5                           293.955551

In [37]:
adolescent_hemoglobin_disparities = pd.DataFrame(
    columns=child_hemoglobin_disparities.columns,
    index=child_hemoglobin_disparities.index,
).droplevel(["age_start", "age_end"])
for group in adolescent_hemoglobin_disparities.index:
    child_values = child_hemoglobin_disparities.droplevel(["age_start", "age_end"]).loc[
        group
    ]
    adult_values = adult_hemoglobin_disparities.droplevel(["age_start", "age_end"]).loc[
        group
    ]
    adolescent_hemoglobin_disparities.loc[group] = (
        child_values * 0.5 + adult_values * 0.5
    ).T

In [38]:
assert adolescent_hemoglobin_disparities.notnull().all().all()
adolescent_hemoglobin_disparities = (
    adolescent_hemoglobin_disparities.reset_index()
    .assign(age_start=5, age_end=15)
    .set_index(list(child_hemoglobin_disparities.index.names))
)

In [39]:
hemoglobin_disparities = pd.concat(
    [
        child_hemoglobin_disparities,
        adolescent_hemoglobin_disparities,
        adult_hemoglobin_disparities,
    ]
)
hemoglobin_disparities

mean         sd  \
sex    age_start age_end pregnant     wealth_quintile                          
Female 0         5       not_pregnant 1                100.844346  19.057214   
                                      2                104.929413  16.388767   
                                      3                108.144135  15.798272   
                                      4                106.092684  14.667368   
                                      5                110.229853  14.279931   
Male   0         5       not_pregnant 1                 99.549536  18.535592   
                                      2                105.371341  16.208981   
                                      3                107.209986  14.817859   
                                      4                 106.62369  17.780204   
                                      5                 108.24703  15.043161   
Female 5         15      not_pregnant 1                112.898306  19.554483   
                                      2                116.611617  16.721486   
                                      3                118.672732  16.180861   
                                      4                118.142864  15.320581   
                                      5                121.106819  14.637155   
Male   5         15      not_pregnant 1                112.250901  19.293672   
                                      2                116.832581  16.631593   
                                      3                118.205657  15.690655   
                                      4                118.408368  16.876999   
                                      5                120.115408  15.018771   
Female 15        125     not_pregnant 1                124.952266  20.051752   
                         pregnant     1                113.492095  22.155564   
                         not_pregnant 2                128.293821  17.054204   
                         pregnant     2                116.926741  19.180831   
                         not_pregnant 3                129.201329  16.563451   
                         pregnant     3                118.841395  14.629155   
                         not_pregnant 4                130.193045  15.973793   
                         pregnant     4                116.357764   15.83606   
                         not_pregnant 5                131.983786   14.99438   
                         pregnant     5                 122.54957  15.731898   
Male   15        125     not_pregnant 1                124.952266  20.051752   
                                      2                128.293821  17.054204   
                                      3                129.201329  16.563451   
                                      4                130.193045  15.973793   
                                      5                131.983786   14.99438   

                                                      effective_sample_size  
sex    age_start age_end pregnant     wealth_quintile                        
Female 0         5       not_pregnant 1                          492.680769  
                                      2                          409.805632  
                                      3                          379.924099  
                                      4                           369.99169  
                                      5                          275.680329  
Male   0         5       not_pregnant 1                          499.512573  
                                      2                          446.163156  
                                      3                          415.615837  
                                      4                          353.032739  
                                      5                          293.955551  
Female 5         15      not_pregnant 1                          800.023496  
                                      2                          722.314627  
        

In [40]:
results_dir = "../results"

In [41]:
hemoglobin_disparities["mean"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/mean_disparities/{location}.csv"
)

In [42]:
hemoglobin_disparities["sd"].rename("value").to_csv(
    f"{results_dir}/hemoglobin/sd_disparities/{location}.csv"
)

## Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at subpopulations which can skew.

In [43]:
group_variables = ["sex", "age_start", "age_end", "pregnant"]

In [44]:
wealth_quintile_probabilities = (
    hhm_data.groupby(group_variables + ["wealth_quintile"], observed=True).weight.sum()
    / hhm_data.groupby(group_variables, observed=True).weight.sum()
)
assert np.allclose(
    wealth_quintile_probabilities.groupby(group_variables, observed=True).sum(), 1.0
)
wealth_quintile_probabilities

sex     age_start  age_end  pregnant      wealth_quintile
Female  0.0        5.0      not_pregnant  1                  0.240568
                                          2                  0.218680
                                          3                  0.206739
                                          4                  0.185403
                                          5                  0.148610
        5.0        15.0     not_pregnant  1                  0.219482
                                          2                  0.209157
                                          3                  0.210067
                                          4                  0.207411
                                          5                  0.153883
        15.0       30.0     not_pregnant  1                  0.154054
                                          2                  0.173552
                                          3                  0.180132
                                

In [45]:
wealth_quintile_probabilities = wealth_quintile_probabilities.unstack()
wealth_quintile_probabilities

wealth_quintile                               1         2         3         4  \
sex    age_start age_end pregnant                                               
Female 0.0       5.0     not_pregnant  0.240568  0.218680  0.206739  0.185403   
       5.0       15.0    not_pregnant  0.219482  0.209157  0.210067  0.207411   
       15.0      30.0    not_pregnant  0.154054  0.173552  0.180132  0.193390   
                         pregnant      0.214447  0.257398  0.173152  0.167345   
       30.0      50.0    not_pregnant  0.176930  0.178372  0.198273  0.205845   
                         pregnant      0.250120  0.185543  0.217769  0.191428   
       50.0      125.0   not_pregnant  0.205491  0.204239  0.195294  0.199461   
Male   0.0       5.0     not_pregnant  0.236076  0.239462  0.208587  0.173674   
       5.0       15.0    not_pregnant  0.226779  0.210595  0.205277  0.206115   
       15.0      30.0    not_pregnant  0.157551  0.173962  0.186354  0.221764   
       30.0      50.0    not_pregnant  0.167606  0.192322  0.201171  0.194831   
       50.0      125.0   not_pregnant  0.189095  0.197845  0.205353  0.217282   

wealth_quintile                               5  
sex    age_start age_end pregnant                
Female 0.0       5.0     not_pregnant  0.148610  
       5.0       15.0    not_pregnant  0.153883  
       15.0      30.0    not_pregnant  0.298871  
                         pregnant      0.187658  
       30.0      50.0    not_pregnant  0.240580  
                         pregnant      0.155140  
       50.0      125.0   not_pregnant  0.195515  
Male   0.0       5.0     not_pregnant  0.142200  
       5.0       15.0    not_pregnant  0.151234  
       15.0      30.0    not_pregnant  0.260370  
       30.0      50.0    not_pregnant  0.244070  
       50.0      125.0   not_pregnant  0.190425

In [46]:
wealth_quintile_probabilities.to_csv(
    f"{results_dir}/wealth_quintile_probabilities/{location}.csv",
)

## LBWSG

### Birth weight

In [47]:
birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace(
    {"not weighed at birth": np.nan, "don't know": np.nan}
).astype(float)

In [48]:
weighted_avg_and_std(birth_data.birth_weight_kilograms, birth_data.weight)

mean                     3309.841385
sd                        945.421146
effective_sample_size     760.932362
dtype: float64

In [49]:
birth_weight_disparities = birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.birth_weight_kilograms, df.weight)
)
birth_weight_disparities

,mean,sd,effective_sample_size
wealth_quintile,,,
1,3282.573946,797.444677,60.479776
2,3192.228089,1065.567197,83.090221
3,3345.650969,1082.086852,87.642164
4,3380.836386,1137.448380,117.887080
5,3305.204538,824.738958,417.886553


In [50]:
birth_weight_disparities = (
    birth_weight_disparities["mean"].rename("value").reset_index()
)
birth_weight_disparities

,wealth_quintile,value
0,1,3282.573946
1,2,3192.228089
2,3,3345.650969
3,4,3380.836386
4,5,3305.204538


In [51]:
birth_weight_disparities.to_csv(
    f"{results_dir}/birth_weight_disparities/{location}.csv", index=False
)

### Short gestation

All we have here is a self-reported duration of pregnancy.

In [52]:
# Basically no difference in mean
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.duration_of_pregnancy, df.weight)
)

,mean,sd,effective_sample_size
wealth_quintile,,,
1,8.995064,0.150729,1368.594233
2,8.988265,0.198151,1086.747517
3,8.991328,0.170364,1015.679732
4,8.992731,0.189762,955.790384
5,8.962728,0.282666,819.804042


In [53]:
birth_data["short_gestation"] = np.where(
    birth_data.duration_of_pregnancy.isnull(),
    np.nan,
    birth_data.duration_of_pregnancy < 9.0,
)

In [54]:
weighted_avg_and_std(birth_data.short_gestation, birth_data.weight)

mean                        0.015970
sd                          0.125360
effective_sample_size    5217.743043
dtype: float64

In [55]:
birth_data.groupby("wealth_quintile").apply(
    lambda df: weighted_avg_and_std(df.short_gestation, df.weight)
)

,mean,sd,effective_sample_size
wealth_quintile,,,
1,0.008170,0.090016,1368.594233
2,0.014857,0.120982,1086.747517
3,0.017128,0.129748,1015.679732
4,0.013500,0.115401,955.790384
5,0.032313,0.176829,819.804042


The trends in short gestation don't make intuitive sense (?), so we do not plan to use them in the sim.